In [0]:
%run ../utils/utils

## Insights do Modelo — Detecção de Anomalias de Vendas

In [0]:
#  Responsabilidade ÚNICA deste notebook: expor os resultados do modelo de
#  forma simples e direta, para consumo do time de negócio e do dashboard
#  no Looker. Não recalcula nada do modelo — só lê o que `modelo.py` já
#  produziu.


import pandas as pd
import matplotlib.pyplot as plt
import pyspark.sql.functions as F
import json

CAMINHO_METADADOS = "IA/encouders/metadados.json"

##  Ler os resultados do modelo

In [0]:
pdf_teste = ler_delta("IA/encouders", "stg_teste_predicoes", STORAGE_OPTIONS).toPandas()

file_client = container_squad1.get_file_client(CAMINHO_METADADOS)
metadados = json.loads(file_client.download_file().readall().decode("utf-8"))

print(f"Total de pedidos avaliados: {len(pdf_teste)}")
print(f"Anomalias encontradas: {pdf_teste['is_anomaly'].sum()} "
      f"({100 * pdf_teste['is_anomaly'].sum() / len(pdf_teste):.2f}%)")
print(f"\nResumo da avaliação do modelo (calculada em modelo.py):")
print(f"  MSE no teste: {metadados.get('avaliacao_mse_teste', 'N/D'):.4f}")
print(f"  Variância explicada aproximada: {metadados.get('avaliacao_variancia_explicada_aprox', 0):.2%}")
print(f"  Features com diferença estatística significativa: "
      f"{metadados.get('avaliacao_qtd_features_significativas', 'N/D')}")

## Segmentação simples de clientes

In [0]:
#  Segmentação de negócio fácil de explicar, baseada no histórico de compras
#  de cada cliente (já calculado no feature engineering):
#  - **Cliente Novo**: nenhum pedido anterior
#  - **Recorrente — Baixo Valor**: já comprou antes, ticket médio abaixo da mediana
#  - **Recorrente — Alto Valor**: já comprou antes, ticket médio acima da mediana

mediana_ticket = pdf_teste.loc[pdf_teste["qtd_pedidos_anteriores_cliente"] > 0, "ticket_medio_historico_cliente"].median()

def classificar_segmento(row):
    if row["qtd_pedidos_anteriores_cliente"] == 0:
        return "Cliente Novo"
    elif row["ticket_medio_historico_cliente"] >= mediana_ticket:
        return "Recorrente - Alto Valor"
    else:
        return "Recorrente - Baixo Valor"

pdf_teste["segmento_cliente"] = pdf_teste.apply(classificar_segmento, axis=1)

resumo_segmento = (
    pdf_teste.groupby("segmento_cliente")
    .agg(qtd_pedidos=("id_pedido", "count"), qtd_anomalias=("is_anomaly", "sum"))
    .assign(taxa_anomalia_pct=lambda df: round(100 * df["qtd_anomalias"] / df["qtd_pedidos"], 2))
    .reset_index()
)

print("===== ANOMALIA POR SEGMENTO DE CLIENTE =====")
display(resumo_segmento)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(resumo_segmento["segmento_cliente"], resumo_segmento["taxa_anomalia_pct"], color="#C44E52")
ax.set_ylabel("% de anomalia")
ax.set_title("Taxa de Anomalia por Segmento de Cliente")
plt.xticks(rotation=15)
for i, v in enumerate(resumo_segmento["taxa_anomalia_pct"]):
    ax.text(i, v, f"{v}%", ha="center", va="bottom", fontweight="bold")
plt.tight_layout()
plt.show()

##  Anomalias x Pedidos Cancelados

In [0]:
#  `status_pedido` nunca foi usado como feature do modelo (para não vazar
#  informação pós-fato) — mas é útil AGORA, para checar se o modelo está
#  identificando pedidos que depois foram cancelados (um sinal de validação
#  de negócio, não do treino).

if "status_pedido" in pdf_teste.columns:
    resumo_status = (
        pdf_teste.groupby("status_pedido")
        .agg(qtd_pedidos=("id_pedido", "count"), qtd_anomalias=("is_anomaly", "sum"))
        .assign(taxa_anomalia_pct=lambda df: round(100 * df["qtd_anomalias"] / df["qtd_pedidos"], 2))
        .reset_index()
        .sort_values("taxa_anomalia_pct", ascending=False)
    )

    print("===== TAXA DE ANOMALIA POR STATUS DO PEDIDO =====")
    display(resumo_status)

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(resumo_status["status_pedido"], resumo_status["taxa_anomalia_pct"], color="#DD8452")
    ax.set_xlabel("% de anomalia")
    ax.set_title("Taxa de Anomalia por Status do Pedido")
    for i, v in enumerate(resumo_status["taxa_anomalia_pct"]):
        ax.text(v, i, f" {v}%", va="center", fontweight="bold")
    plt.tight_layout()
    plt.show()

    if "Cancelado" in resumo_status["status_pedido"].values:
        taxa_cancelado = resumo_status.loc[resumo_status["status_pedido"] == "Cancelado", "taxa_anomalia_pct"].iloc[0]
        taxa_geral = round(100 * pdf_teste["is_anomaly"].sum() / len(pdf_teste), 2)
        print(f"\nPedidos cancelados têm taxa de anomalia de {taxa_cancelado}%, "
              f"contra {taxa_geral}% da base geral "
              f"({'ACIMA' if taxa_cancelado > taxa_geral else 'ABAIXO'} da média).")
else:
    print("Coluna 'status_pedido' não encontrada — rode o modelo.py atualizado antes.")


##  Comparação Squad1 x Squad3

In [0]:
#  Squad1 e Squad3 alimentam o mesmo modelo (união feita no feature_engineer.py),
#  mas cada squad pode ter um perfil de vendas diferente. Aqui comparamos a
#  taxa de anomalia e o tipo de anomalia mais comum entre as duas origens —
#  útil para saber se um dos squads está gerando dados "mais estranhos" que
#  o outro (pode indicar problema de qualidade de dados específico daquele
#  squad, e não necessariamente uma anomalia de negócio real).

if "origem_squad" in pdf_teste.columns:
    resumo_squad = (
        pdf_teste.groupby("origem_squad")
        .agg(qtd_pedidos=("id_pedido", "count"), qtd_anomalias=("is_anomaly", "sum"))
        .assign(taxa_anomalia_pct=lambda df: round(100 * df["qtd_anomalias"] / df["qtd_pedidos"], 2))
        .reset_index()
        .sort_values("taxa_anomalia_pct", ascending=False)
    )

    print("===== TAXA DE ANOMALIA POR ORIGEM (SQUAD1 x SQUAD3) =====")
    display(resumo_squad)

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(resumo_squad["origem_squad"], resumo_squad["taxa_anomalia_pct"], color=["#4C72B0", "#DD8452"])
    ax.set_ylabel("% de anomalia")
    ax.set_title("Taxa de Anomalia: Squad1 x Squad3")
    for i, v in enumerate(resumo_squad["taxa_anomalia_pct"]):
        ax.text(i, v, f"{v}%", ha="center", va="bottom", fontweight="bold")
    plt.tight_layout()
    plt.show()

    if len(resumo_squad) == 2:
        maior = resumo_squad.iloc[0]
        menor = resumo_squad.iloc[1]
        print(f"\n{maior['origem_squad']} tem taxa de anomalia de {maior['taxa_anomalia_pct']}%, "
              f"contra {menor['taxa_anomalia_pct']}% de {menor['origem_squad']}.")

    # Tipo de anomalia mais comum, separado por squad — mostra se cada squad
    # "quebra" o modelo por um motivo diferente.
    if "feature_dominante" in pdf_teste.columns:
        anomalias_por_squad = pdf_teste[pdf_teste["is_anomaly"]]
        if len(anomalias_por_squad) > 0:
            top_feature_por_squad = (
                anomalias_por_squad.groupby(["origem_squad", "feature_dominante"])
                .size()
                .reset_index(name="qtd")
                .sort_values(["origem_squad", "qtd"], ascending=[True, False])
                .groupby("origem_squad")
                .head(3)
            )
            print("\n===== TOP 3 TIPOS DE ANOMALIA POR ORIGEM =====")
            display(top_feature_por_squad)
else:
    print("Coluna 'origem_squad' não encontrada — rode o feature_engineer.py/modelo.py atualizados antes.")


##  Por que essas anomalias aconteceram — produtos comprados


In [0]:
#  `feature_dominante` diz QUAL FEATURE NUMÉRICA (ex: "razao_frete_valor")
#  mais contribuiu para o erro de reconstrução — mas não diz O QUE a pessoa
#  comprou. Para responder ao cliente/professor "por que esse pedido foi
#  sinalizado" de forma concreta (ex: "porque essa combinação de produtos é
#  rara" — o clássico caso de mercado "cerveja + fralda"), voltamos aos
#  ITENS de cada pedido anômalo e olhamos as categorias envolvidas.
#
#  Ajuste os nomes abaixo se o schema real de `ecommerce_categorias` /
#  `ecommerce_produtos` usar nomes de coluna diferentes.
COLUNA_ID_CATEGORIA = "id_categoria"
COLUNA_NOME_CATEGORIA = "nome_categoria"


def ler_delta_squad3_insights(camada, tabela, storage_opts):
    """Réplica do helper usado em feature_engineer.py — necessária aqui porque
    insights.py roda de forma independente (não importa o notebook de FE)."""
    caminho_final = get_delta_path_squad3(camada, tabela, storage_opts)
    account_name = storage_opts.get("account_name")
    client_id = storage_opts.get("client_id")
    client_secret = storage_opts.get("client_secret")
    tenant_id = storage_opts.get("tenant_id")

    return (spark.read
        .format("delta")
        .option(f"fs.azure.account.auth.type.{account_name}.dfs.core.windows.net", "OAuth")
        .option(f"fs.azure.account.oauth.provider.type.{account_name}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
        .option(f"fs.azure.account.oauth2.client.id.{account_name}.dfs.core.windows.net", client_id)
        .option(f"fs.azure.account.oauth2.client.secret.{account_name}.dfs.core.windows.net", client_secret)
        .option(f"fs.azure.account.oauth2.client.endpoint.{account_name}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
        .load(caminho_final))


ANALISE_PRODUTOS_DISPONIVEL = False
try:
    # Reconstrói, a partir da Silver bruta, o mapa id_pedido -> categorias
    # compradas (mesmo esquema de prefixo s1_/s3_ do feature_engineer.py,
    # para bater com os IDs que já estão em pdf_teste).
    df_itens_s1 = (ler_delta("silver", "ecommerce_itens_pedido", STORAGE_OPTIONS)
                   .select("id_pedido", "sku")
                   .withColumn("id_pedido", F.concat(F.lit("s1_"), F.col("id_pedido").cast("string")))
                   .withColumn("sku", F.concat(F.lit("s1_"), F.col("sku"))))
    df_itens_s3 = (ler_delta_squad3_insights("silver", "ecommerce_itens_pedido", STORAGE_OPTIONS)
                   .select("id_pedido", "sku")
                   .withColumn("id_pedido", F.concat(F.lit("s3_"), F.col("id_pedido").cast("string")))
                   .withColumn("sku", F.concat(F.lit("s3_"), F.col("sku"))))
    df_itens_todos = df_itens_s1.unionByName(df_itens_s3)

    df_produtos_s1 = (ler_delta("silver", "ecommerce_produtos", STORAGE_OPTIONS)
                       .select("sku", COLUNA_ID_CATEGORIA)
                       .withColumn("sku", F.concat(F.lit("s1_"), F.col("sku")))
                       .withColumn(COLUNA_ID_CATEGORIA, F.concat(F.lit("s1_"), F.col(COLUNA_ID_CATEGORIA).cast("string"))))
    df_produtos_s3 = (ler_delta_squad3_insights("silver", "ecommerce_produtos", STORAGE_OPTIONS)
                       .select("sku", COLUNA_ID_CATEGORIA)
                       .withColumn("sku", F.concat(F.lit("s3_"), F.col("sku")))
                       .withColumn(COLUNA_ID_CATEGORIA, F.concat(F.lit("s3_"), F.col(COLUNA_ID_CATEGORIA).cast("string"))))
    df_produtos_todos = df_produtos_s1.unionByName(df_produtos_s3)

    df_categorias_s1 = (ler_delta("silver", "ecommerce_categorias", STORAGE_OPTIONS)
                         .select(COLUNA_ID_CATEGORIA, COLUNA_NOME_CATEGORIA)
                         .withColumn(COLUNA_ID_CATEGORIA, F.concat(F.lit("s1_"), F.col(COLUNA_ID_CATEGORIA).cast("string"))))
    df_categorias_s3 = (ler_delta_squad3_insights("silver", "ecommerce_categorias", STORAGE_OPTIONS)
                         .select(COLUNA_ID_CATEGORIA, COLUNA_NOME_CATEGORIA)
                         .withColumn(COLUNA_ID_CATEGORIA, F.concat(F.lit("s3_"), F.col(COLUNA_ID_CATEGORIA).cast("string"))))
    df_categorias_todos = df_categorias_s1.unionByName(df_categorias_s3).dropDuplicates([COLUNA_ID_CATEGORIA])

    df_pedido_categoria = (
        df_itens_todos
        .join(df_produtos_todos, "sku", "left")
        .join(df_categorias_todos, COLUNA_ID_CATEGORIA, "left")
        .select("id_pedido", COLUNA_NOME_CATEGORIA)
        .dropna(subset=[COLUNA_NOME_CATEGORIA])
        .distinct()
    )

    pdf_pedido_categoria = df_pedido_categoria.toPandas()
    ANALISE_PRODUTOS_DISPONIVEL = True

except Exception as e:
    print(f"[Aviso] Não foi possível montar a análise de produtos: {e}")
    print("Confira se COLUNA_ID_CATEGORIA / COLUNA_NOME_CATEGORIA batem com o "
          "schema real de 'ecommerce_categorias' / 'ecommerce_produtos'.")


###  Gasto médio e categorias sobre-representadas nas anomalias

In [0]:

if ANALISE_PRODUTOS_DISPONIVEL:
    ids_anomalos = set(pdf_teste.loc[pdf_teste["is_anomaly"], "id_pedido"])
    ids_normais = set(pdf_teste.loc[~pdf_teste["is_anomaly"], "id_pedido"])

    ticket_medio_anomalia = pdf_teste.loc[pdf_teste["is_anomaly"], "valor_total"].mean()
    ticket_medio_normal = pdf_teste.loc[~pdf_teste["is_anomaly"], "valor_total"].mean()

    print(f"Ticket médio — Anomalia: R$ {ticket_medio_anomalia:.2f} | Normal: R$ {ticket_medio_normal:.2f} "
          f"({'ACIMA' if ticket_medio_anomalia > ticket_medio_normal else 'ABAIXO'} da média normal)")

    cat_anomalia = pdf_pedido_categoria[pdf_pedido_categoria["id_pedido"].isin(ids_anomalos)]
    cat_normal = pdf_pedido_categoria[pdf_pedido_categoria["id_pedido"].isin(ids_normais)]

    freq_anomalia = (cat_anomalia[COLUNA_NOME_CATEGORIA].value_counts() / max(len(ids_anomalos), 1) * 100).round(1)
    freq_normal = (cat_normal[COLUNA_NOME_CATEGORIA].value_counts() / max(len(ids_normais), 1) * 100).round(1)

    resumo_categorias = pd.DataFrame({
        "pct_pedidos_anomalos": freq_anomalia,
        "pct_pedidos_normais": freq_normal,
    }).fillna(0.0)
    # +0.01 evita divisão por zero sem distorcer o ranking
    resumo_categorias["lift"] = (
        (resumo_categorias["pct_pedidos_anomalos"] + 0.01) /
        (resumo_categorias["pct_pedidos_normais"] + 0.01)
    ).round(2)
    resumo_categorias = resumo_categorias.sort_values("lift", ascending=False)

    print("\n===== CATEGORIAS SOBRE-REPRESENTADAS NAS ANOMALIAS =====")
    print("(lift > 1: a categoria aparece proporcionalmente mais em pedidos anômalos que em normais)")
    display(resumo_categorias.head(10))

###  Combinações de categorias nas anomalias (ex: "cerveja + fralda")

In [0]:
#  Regra de associação simplificada (sem lib externa): para cada par de
#  categorias que aparece junto num mesmo pedido anômalo, calcula suporte
#  (% dos pedidos anômalos com as duas juntas), confiança (dado que
#  comprou A, qual % também comprou B) e lift (se a combinação é mais
#  frequente do que se A e B fossem independentes). Lift >> 1 é o sinal
#  clássico de "combinação estranha, mas real" — a mesma lógica por trás
#  do achado de mercado "cerveja + fralda".
 
if ANALISE_PRODUTOS_DISPONIVEL:
    from itertools import combinations
 
    cestas_anomalia = cat_anomalia.groupby("id_pedido")[COLUNA_NOME_CATEGORIA].apply(set)
    total_cestas = len(cestas_anomalia)
    contagem_individual = cat_anomalia[COLUNA_NOME_CATEGORIA].value_counts().to_dict()
 
    contagem_pares = {}
    for cesta in cestas_anomalia:
        for par in combinations(sorted(cesta), 2):
            contagem_pares[par] = contagem_pares.get(par, 0) + 1
 
    linhas_regras = []
    for (cat_a, cat_b), qtd_junto in contagem_pares.items():
        suporte = qtd_junto / total_cestas
        confianca_a_b = qtd_junto / contagem_individual[cat_a]
        prob_b = contagem_individual[cat_b] / total_cestas
        lift = (confianca_a_b / prob_b) if prob_b > 0 else 0
        linhas_regras.append({
            "categoria_a": cat_a, "categoria_b": cat_b,
            "qtd_pedidos_juntos": qtd_junto,
            "suporte_pct": round(100 * suporte, 2),
            "confianca_pct": round(100 * confianca_a_b, 2),
            "lift": round(lift, 2),
        })
 
    if linhas_regras:
        df_regras = pd.DataFrame(linhas_regras).sort_values("lift", ascending=False)
        print("===== TOP 10 COMBINAÇÕES DE CATEGORIAS MAIS FORTES NAS ANOMALIAS =====")
        display(df_regras.head(10))
 
        top_regra = df_regras.iloc[0]
        print(f"\nCombinação mais forte: '{top_regra['categoria_a']}' + '{top_regra['categoria_b']}' "
              f"aparece em {top_regra['qtd_pedidos_juntos']} pedidos anômalos (lift {top_regra['lift']}x) — "
              f"esse é o tipo de resposta concreta para o cliente: 'esse pedido foi sinalizado porque "
              f"combina X e Y de um jeito incomum', análogo ao caso clássico de cerveja + fralda.")
    else:
        print("Nenhum pedido anômalo tem 2+ categorias distintas para formar combinações.")

##  Tipos de anomalia mais comuns (o que mais chama atenção nas compras estranhas)

In [0]:
 
anomalias = pdf_teste[pdf_teste["is_anomaly"]]
 
if "feature_dominante" in pdf_teste.columns and len(anomalias) > 0:
    contagem_tipos = anomalias["feature_dominante"].value_counts().reset_index()
    contagem_tipos.columns = ["tipo_anomalia", "qtd_anomalias"]
    contagem_tipos["pct"] = (100 * contagem_tipos["qtd_anomalias"] / contagem_tipos["qtd_anomalias"].sum()).round(1)
 
    print("===== TIPOS DE ANOMALIA MAIS COMUNS =====")
    display(contagem_tipos)
 
    # Gráfico de pizza com muitas fatias pequenas fica ilegível (rótulos se
    # sobrepõem). Aqui: agrupa fatias residuais (< LIMIAR_OUTROS_PCT) em
    # "Outros" e usa barras horizontais ordenadas — muito mais legível quando
    # há uma feature dominante grande e várias pequenas.
    LIMIAR_OUTROS_PCT = 2.0
 
    principais = contagem_tipos[contagem_tipos["pct"] >= LIMIAR_OUTROS_PCT].copy()
    residual = contagem_tipos[contagem_tipos["pct"] < LIMIAR_OUTROS_PCT]
 
    if len(residual) > 0:
        linha_outros = pd.DataFrame([{
            "tipo_anomalia": f"Outros ({len(residual)} features)",
            "qtd_anomalias": residual["qtd_anomalias"].sum(),
            "pct": round(residual["pct"].sum(), 1),
        }])
        contagem_plot = pd.concat([principais, linha_outros], ignore_index=True)
    else:
        contagem_plot = principais
 
    contagem_plot = contagem_plot.sort_values("qtd_anomalias", ascending=True)
 
    fig, ax = plt.subplots(figsize=(9, max(4, 0.5 * len(contagem_plot))))
    barras = ax.barh(contagem_plot["tipo_anomalia"], contagem_plot["pct"], color="#4C72B0")
    ax.set_xlabel("% das anomalias")
    ax.set_title("Distribuição dos Tipos de Anomalia (feature dominante)")
    for i, v in enumerate(contagem_plot["pct"]):
        ax.text(v, i, f" {v}%", va="center", fontweight="bold")
    ax.set_xlim(0, contagem_plot["pct"].max() * 1.15)
    plt.tight_layout()
    plt.show()
 
    if len(residual) > 0:
        print(f"'Outros' agrupa as {len(residual)} features com menos de {LIMIAR_OUTROS_PCT}% cada: "
              f"{', '.join(residual['tipo_anomalia'].tolist())}")

## Exemplos concretos — as anomalias mais fortes

In [0]:
colunas_exemplo = ["id_pedido", "id_cliente", "segmento_cliente", "status_pedido",
                   "valor_total", "hora_do_dia", "feature_dominante", "erro_reconstrucao"]
colunas_exemplo_disp = [c for c in colunas_exemplo if c in pdf_teste.columns]
 
print("===== TOP 15 ANOMALIAS MAIS FORTES =====")
display(
    pdf_teste[pdf_teste["is_anomaly"]]
    .sort_values("erro_reconstrucao", ascending=False)
    .head(15)[colunas_exemplo_disp]
)


## Salvar tabela final — pronta para o Looker

In [0]:

import pyspark.sql.functions as F

# Score 0-100, mais intuitivo para quem for consumir no dashboard
score_min = pdf_teste["erro_reconstrucao"].min()
score_max = pdf_teste["erro_reconstrucao"].max()
pdf_teste["anomaly_score_0_100"] = (
    100 * (pdf_teste["erro_reconstrucao"] - score_min) / (score_max - score_min)
).round(2)

colunas_looker = ["id_pedido", "id_cliente", "segmento_cliente", "status_pedido",
                   "origem_squad", "valor_total", "hora_do_dia",
                   "feature_dominante", "anomaly_score_0_100", "is_anomaly"]
colunas_looker = [c for c in colunas_looker if c in pdf_teste.columns]

df_gold_insights = spark.createDataFrame(pdf_teste[colunas_looker]) \
    .withColumn("data_processamento", F.current_timestamp())

gravar_delta(
    df=df_gold_insights,
    camada="gold",
    tabela="gold_insights_anomalias",
    storage_opts=STORAGE_OPTIONS,
    mode="overwrite",
    particionar=False
)

try:
    escrever_sqlserver_gold(df_spark=df_gold_insights, schema="squad1",
                             tabela="gold_insights_anomalias_encouder", modo="overwrite")
    print("[Sucesso] gold_insights_anomalias sincronizada no SQL Server (Looker consome daqui).")
except Exception as e:
    print(f"[Aviso] Não foi possível sincronizar com o SQL Server: {e}")

print(f"\ngold_insights_anomalias gravada: {df_gold_insights.count()} linhas.")


## Publicar tabelas-resumo para o Looker (SQL Server)

In [0]:

#  `gold_insights_anomalias` tem granularidade de PEDIDO — ótima para uma
#  tabela detalhada/drill-down no Looker, mas ruim como fonte direta de
#  gráfico de barra/pizza (o Looker teria que agregar toda vez, e cada
#  gráfico recalcularia a mesma coisa). Por isso publicamos também as
#  tabelas JÁ AGREGADAS calculadas acima — uma por pergunta de negócio —
#  como fontes de dado próprias no Looker. Todas em modo "overwrite":
#  cada execução substitui o snapshot anterior (evita o problema de
#  métricas duplicando por causa de modo append + reprocessamento, que já
#  intercorreu no dashboard de volumetria).

def publicar_gold_looker(pdf: pd.DataFrame, tabela: str, indice_para_coluna: str = None):
    """Grava um DataFrame Pandas já agregado como tabela Gold (Delta) e
    espelha no SQL Server, pronto para virar fonte de dado no Looker."""
    pdf_pub = pdf.reset_index() if indice_para_coluna else pdf.copy()
    if indice_para_coluna and pdf_pub.columns[0] != indice_para_coluna:
        pdf_pub = pdf_pub.rename(columns={pdf_pub.columns[0]: indice_para_coluna})

    df_spark_pub = spark.createDataFrame(pdf_pub).withColumn("data_processamento", F.current_timestamp())

    gravar_delta(df=df_spark_pub, camada="gold", tabela=tabela,
                 storage_opts=STORAGE_OPTIONS, mode="overwrite", particionar=False)
    try:
        escrever_sqlserver_gold(df_spark=df_spark_pub, schema="squad1", tabela=tabela, modo="overwrite")
        print(f"[Sucesso] {tabela}: {df_spark_pub.count()} linhas -> Delta + SQL Server.")
    except Exception as e:
        print(f"[Aviso] {tabela} gravada no Delta, mas falhou no SQL Server: {e}")


publicar_gold_looker(resumo_segmento, "gold_encouder_segmento_cliente")

if "resumo_status" in globals():
    publicar_gold_looker(resumo_status, "gold_encouder_status_pedido")

if "resumo_squad" in globals():
    publicar_gold_looker(resumo_squad, "gold_encouder_squad")

if "contagem_tipos" in globals():
    publicar_gold_looker(contagem_tipos, "gold_encouder_feature_dominante")

if "resumo_categorias" in globals():
    publicar_gold_looker(resumo_categorias, "gold_categorias_anomalias_encouder", indice_para_coluna="categoria")

if "df_regras" in globals():
    publicar_gold_looker(df_regras, "gold_regras_associacao_categorias_encouder")

print("\nTabelas-resumo publicadas. No Looker, use cada uma como fonte de dado separada "
      "(não recalcule agregações lá — elas já vêm prontas daqui).")